In [2]:
# ============================================================
# FULL PAPER-READY PIPELINE
# Indonesian Hate Speech Detection
# - Classical ML
# - IndoBERT
# - XLM-R
# - Weighted loss
# - Oversampling
# - Threshold tuning
# - Agreement-based pseudo-labeling
# - Publication figures
# ============================================================

import os
import re
import math
import json
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from dataclasses import dataclass

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.sparse import hstack, csr_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

warnings.filterwarnings("ignore")

# ============================================================
# 1. GLOBALS
# ============================================================

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "/net/pr2/projects/plgrid/plggman01/HateSpeech"
OUTPUT_DIR = "./paper_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PATH_IDHSD = os.path.join(DATA_DIR, "IDHSD_RIO_unbalanced_713_2017.txt")
PATH_572 = os.path.join(DATA_DIR, "572-hate-speech-dataset.csv")
PATH_RE = os.path.join(DATA_DIR, "re_dataset.csv")
PATH_KAMUSALAY = os.path.join(DATA_DIR, "new_kamusalay.csv")
PATH_ABUSIVE = os.path.join(DATA_DIR, "abusive.csv")

print("Using device:", DEVICE)

# ============================================================
# 2. FILE LOADING
# ============================================================

def safe_read_csv(path, sep=",", header="infer"):
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]
    last_error = None
    for enc in encodings:
        try:
            return pd.read_csv(path, sep=sep, encoding=enc, header=header)
        except Exception as e:
            last_error = e
    raise ValueError(f"Could not read file {path}. Last error: {last_error}")


def normalize_binary_label(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"hs", "hate", "hate_speech", "1", "true", "yes"}:
            return 1
        if v in {"non_hs", "non-hs", "non hs", "not_hs", "0", "false", "no"}:
            return 0

    if isinstance(value, (int, float, np.integer, np.floating)):
        return int(value)

    return np.nan


def print_dataset_summary(df, name="dataset"):
    print("\n" + "=" * 70)
    print(f"DATASET: {name}")
    print("=" * 70)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    if "label" in df.columns:
        print("Label distribution:", Counter(df["label"].tolist()))


def load_idhsd_dataset(path):
    df = safe_read_csv(path, sep="\t")
    df = df.rename(columns={"Label": "raw_label", "Tweet": "text"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "IDHSD_713"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df


def load_572_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["comment_text", "tweet", "Tweet", "text", "Text"]
    possible_label_cols = ["class", "label", "Label", "HS"]

    text_col = next((c for c in possible_text_cols if c in df.columns), None)
    label_col = next((c for c in possible_label_cols if c in df.columns), None)

    if text_col is None or label_col is None:
        raise ValueError(f"Could not detect text/label columns in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "HS_572"
    df = df[["text", "label", "source"]].dropna().reset_index(drop=True)
    return df


def load_re_dataset(path):
    df = safe_read_csv(path)

    possible_text_cols = ["Tweet", "tweet", "text", "Text"]
    text_col = next((c for c in possible_text_cols if c in df.columns), None)

    if text_col is None:
        raise ValueError(f"No text column found in {path}")

    if "HS" in df.columns:
        label_col = "HS"
    elif "label" in df.columns:
        label_col = "label"
    else:
        raise ValueError(f"No HS/label column found in {path}")

    df = df.rename(columns={text_col: "text", label_col: "raw_label"})
    df["label"] = df["raw_label"].apply(normalize_binary_label)
    df["source"] = "RE_13169"

    keep_cols = ["text", "label", "source"]
    if "Abusive" in df.columns:
        keep_cols.append("Abusive")

    df = df[keep_cols].dropna(subset=["text", "label"]).reset_index(drop=True)
    return df


def merge_datasets(datasets):
    merged = pd.concat(datasets, axis=0, ignore_index=True)
    merged["text"] = merged["text"].astype(str).str.strip()
    merged = merged[merged["text"].str.len() > 0].copy()
    merged = merged.drop_duplicates(subset=["text"]).reset_index(drop=True)
    return merged

# ============================================================
# 3. TEXT NORMALIZATION
# ============================================================

URL_PATTERN = re.compile(r"http\S+|www\S+|https\S+")
MENTION_PATTERN = re.compile(r"@\w+")
HASHTAG_PATTERN = re.compile(r"#(\w+)")
NON_ALNUM_PATTERN = re.compile(r"[^a-zA-Z0-9\s]")
MULTISPACE_PATTERN = re.compile(r"\s+")
REPEAT_CHAR_PATTERN = re.compile(r"(.)\1{2,}")


def load_kamusalay_dict(path):
    df = safe_read_csv(path)
    if df.shape[1] < 2:
        raise ValueError("new_kamusalay.csv must have at least 2 columns.")

    col1, col2 = df.columns[:2]
    slang_dict = {}
    for _, row in df.iterrows():
        src = str(row[col1]).strip().lower()
        tgt = str(row[col2]).strip().lower()
        if src and src != "nan":
            slang_dict[src] = tgt
    return slang_dict


def load_abusive_lexicon(path):
    df = safe_read_csv(path)
    words = set()
    first_col = df.columns[0]
    for item in df[first_col].dropna().tolist():
        words.add(str(item).strip().lower())
    return words


def reduce_repeated_chars(text):
    return REPEAT_CHAR_PATTERN.sub(r"\1\1", text)


def split_hashtag(match):
    token = match.group(1)
    return f" {token} "


def basic_clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = text.replace("\\n", " ")
    text = text.replace("\\/", "/")
    text = reduce_repeated_chars(text)
    text = URL_PATTERN.sub(" ", text)
    text = MENTION_PATTERN.sub(" ", text)
    text = HASHTAG_PATTERN.sub(split_hashtag, text)
    text = NON_ALNUM_PATTERN.sub(" ", text)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()

    return text


def slang_normalize(text, slang_dict):
    tokens = text.split()
    normalized = [slang_dict.get(tok, tok) for tok in tokens]
    return " ".join(normalized)


def preprocess_text(text, slang_dict=None):
    text = basic_clean_text(text)
    if slang_dict is not None:
        text = slang_normalize(text, slang_dict)
    text = MULTISPACE_PATTERN.sub(" ", text).strip()
    return text


def prepare_full_dataset():
    slang_dict = load_kamusalay_dict(PATH_KAMUSALAY)
    abusive_words = load_abusive_lexicon(PATH_ABUSIVE)

    ds_idhsd = load_idhsd_dataset(PATH_IDHSD)
    ds_572 = load_572_dataset(PATH_572)
    ds_re = load_re_dataset(PATH_RE)

    print_dataset_summary(ds_idhsd, "IDHSD")
    print_dataset_summary(ds_572, "HS_572")
    print_dataset_summary(ds_re, "RE_13169")

    data = merge_datasets([ds_idhsd, ds_572, ds_re])

    data["clean_text"] = data["text"].apply(lambda x: preprocess_text(x, slang_dict))
    data = data[data["clean_text"].str.len() > 2].copy()

    data["abusive_count"] = data["clean_text"].apply(
        lambda txt: sum(1 for tok in txt.split() if tok in abusive_words)
    )

    data["length_words"] = data["clean_text"].apply(lambda x: len(x.split()))
    data["length_chars"] = data["clean_text"].apply(len)

    print_dataset_summary(
        data[["clean_text", "label"]].rename(columns={"clean_text": "text"}),
        "Merged final"
    )
    return data

# ============================================================
# 4. SPLITS + BALANCING
# ============================================================

def make_train_val_test_split(data, test_size=0.2, val_size=0.1):
    train_df, test_df = train_test_split(
        data,
        test_size=test_size,
        random_state=RANDOM_STATE,
        stratify=data["label"]
    )

    train_df, val_df = train_test_split(
        train_df,
        test_size=val_size,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )

    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)


def make_low_label_split(train_df, labeled_fraction=0.05):
    labeled_df, unlabeled_df = train_test_split(
        train_df,
        train_size=labeled_fraction,
        random_state=RANDOM_STATE,
        stratify=train_df["label"]
    )
    return labeled_df.reset_index(drop=True), unlabeled_df.reset_index(drop=True)


def oversample_minority_df(df, label_col="label"):
    counts = df[label_col].value_counts()
    if len(counts) < 2:
        return df.copy()

    max_count = counts.max()
    pieces = []
    for cls, group in df.groupby(label_col):
        if len(group) < max_count:
            add = group.sample(max_count - len(group), replace=True, random_state=RANDOM_STATE)
            group = pd.concat([group, add], ignore_index=True)
        pieces.append(group)

    out = pd.concat(pieces, ignore_index=True).sample(frac=1.0, random_state=RANDOM_STATE)
    return out.reset_index(drop=True)

# ============================================================
# 5. CLASSICAL FEATURES + MODELS
# ============================================================

class TfidfFusionClassifier(BaseEstimator, ClassifierMixin):
    """
    Word + char TF-IDF fusion with a backend classifier.
    """
    def __init__(
        self,
        clf_name="logreg",
        max_word_features=30000,
        max_char_features=20000
    ):
        self.clf_name = clf_name
        self.max_word_features = max_word_features
        self.max_char_features = max_char_features

        self.word_vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_features=max_word_features,
            sublinear_tf=True
        )
        self.char_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_features=max_char_features,
            sublinear_tf=True
        )
        self.clf = None
        self.threshold_ = 0.5

    def _build_clf(self):
        if self.clf_name == "svm":
            return LinearSVC(class_weight="balanced", random_state=RANDOM_STATE)
        if self.clf_name == "logreg":
            return LogisticRegression(
                max_iter=400,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        if self.clf_name == "nb":
            return MultinomialNB()
        if self.clf_name == "dt":
            return DecisionTreeClassifier(
                max_depth=40,
                min_samples_split=10,
                class_weight="balanced",
                random_state=RANDOM_STATE
            )
        raise ValueError(f"Unknown clf_name: {self.clf_name}")

    def fit(self, X, y):
        Xw = self.word_vectorizer.fit_transform(X)
        Xc = self.char_vectorizer.fit_transform(X)
        Xf = hstack([Xw, Xc]).tocsr()

        self.clf = self._build_clf()
        self.clf.fit(Xf, y)
        return self

    def _transform(self, X):
        Xw = self.word_vectorizer.transform(X)
        Xc = self.char_vectorizer.transform(X)
        return hstack([Xw, Xc]).tocsr()

    def predict_proba_like(self, X):
        Xf = self._transform(X)

        if hasattr(self.clf, "predict_proba"):
            return self.clf.predict_proba(Xf)[:, 1]

        if hasattr(self.clf, "decision_function"):
            scores = self.clf.decision_function(Xf)
            scores = np.asarray(scores).reshape(-1)
            probs = 1 / (1 + np.exp(-scores))
            return probs

        preds = self.clf.predict(Xf)
        return np.asarray(preds).astype(float)

    def predict(self, X):
        probs = self.predict_proba_like(X)
        return (probs >= self.threshold_).astype(int)

    def set_threshold(self, threshold):
        self.threshold_ = threshold
        return self


def build_classical_models():
    return {
        "fusion_svm": TfidfFusionClassifier(clf_name="svm"),
        "fusion_logreg": TfidfFusionClassifier(clf_name="logreg"),
        "fusion_nb": TfidfFusionClassifier(clf_name="nb"),
        "fusion_dt": TfidfFusionClassifier(clf_name="dt"),
    }

# ============================================================
# 6. METRICS + THRESHOLD TUNING
# ============================================================

def evaluate_predictions(y_true, y_pred, y_prob=None, model_name="model"):
    result = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

    if y_prob is not None:
        try:
            result["roc_auc"] = roc_auc_score(y_true, y_prob)
        except Exception:
            result["roc_auc"] = np.nan
    else:
        result["roc_auc"] = np.nan

    return result


def print_evaluation(y_true, y_pred, model_name="model"):
    print("\n" + "=" * 70)
    print(f"RESULTS: {model_name}")
    print("=" * 70)
    print(classification_report(y_true, y_pred, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))


def tune_threshold_from_probs(y_true, y_prob, metric="macro_f1"):
    best_thr = 0.5
    best_score = -1.0

    for thr in np.arange(0.20, 0.81, 0.02):
        y_pred = (y_prob >= thr).astype(int)

        if metric == "macro_f1":
            score = f1_score(y_true, y_pred, average="macro")
        elif metric == "accuracy":
            score = accuracy_score(y_true, y_pred)
        else:
            raise ValueError("Unsupported metric.")

        if score > best_score:
            best_score = score
            best_thr = float(thr)

    return best_thr, best_score

# ============================================================
# 7. CLASSICAL TRAINING
# ============================================================

def run_classical_baselines(labeled_df, val_df, test_df, scenario_name):
    models = build_classical_models()
    results = []
    fitted_models = {}

    # oversample only labeled set
    labeled_balanced = oversample_minority_df(labeled_df)

    for name, model in models.items():
        print(f"\nTraining classical model: {name}")
        model.fit(labeled_balanced["clean_text"], labeled_balanced["label"])

        val_prob = model.predict_proba_like(val_df["clean_text"])
        best_thr, _ = tune_threshold_from_probs(val_df["label"], val_prob, metric="macro_f1")
        model.set_threshold(best_thr)

        val_pred = model.predict(val_df["clean_text"])
        test_prob = model.predict_proba_like(test_df["clean_text"])
        test_pred = model.predict(test_df["clean_text"])

        print(f"Best validation threshold for {name}: {best_thr:.2f}")

        print_evaluation(test_df["label"], test_pred, model_name=f"{name}_{scenario_name}_test")

        val_result = evaluate_predictions(
            val_df["label"], val_pred, val_prob, model_name=f"{name}"
        )
        test_result = evaluate_predictions(
            test_df["label"], test_pred, test_prob, model_name=f"{name}"
        )

        val_result.update({"split": "val", "scenario": scenario_name, "best_threshold": best_thr})
        test_result.update({"split": "test", "scenario": scenario_name, "best_threshold": best_thr})

        results.extend([val_result, test_result])
        fitted_models[name] = model

    # simple soft-like voting via averaged probabilities of best 3
    ensemble_names = ["fusion_svm", "fusion_logreg", "fusion_nb"]
    ensemble_probs_val = []
    ensemble_probs_test = []

    for n in ensemble_names:
        ensemble_probs_val.append(fitted_models[n].predict_proba_like(val_df["clean_text"]))
        ensemble_probs_test.append(fitted_models[n].predict_proba_like(test_df["clean_text"]))

    avg_val_prob = np.mean(np.vstack(ensemble_probs_val), axis=0)
    avg_test_prob = np.mean(np.vstack(ensemble_probs_test), axis=0)

    ens_thr, _ = tune_threshold_from_probs(val_df["label"], avg_val_prob, metric="macro_f1")
    ens_val_pred = (avg_val_prob >= ens_thr).astype(int)
    ens_test_pred = (avg_test_prob >= ens_thr).astype(int)

    print(f"\nEnsemble threshold: {ens_thr:.2f}")
    print_evaluation(test_df["label"], ens_test_pred, model_name=f"fusion_ensemble_{scenario_name}_test")

    val_result = evaluate_predictions(val_df["label"], ens_val_pred, avg_val_prob, model_name="fusion_ensemble")
    test_result = evaluate_predictions(test_df["label"], ens_test_pred, avg_test_prob, model_name="fusion_ensemble")
    val_result.update({"split": "val", "scenario": scenario_name, "best_threshold": ens_thr})
    test_result.update({"split": "test", "scenario": scenario_name, "best_threshold": ens_thr})

    results.extend([val_result, test_result])

    return pd.DataFrame(results), fitted_models

# ============================================================
# 8. TRANSFORMER DATASET
# ============================================================

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding=True,
            max_length=max_length
        )
        self.labels = labels.tolist() if isinstance(labels, pd.Series) else list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# ============================================================
# 9. TRANSFORMER TRAINER WITH WEIGHTED LOSS
# ============================================================

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        if self.class_weights is not None:
            class_weights = self.class_weights.to(logits.device)
            loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        else:
            loss_fct = nn.CrossEntropyLoss()
        num_labels = logits.size(-1)
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


def compute_metrics_for_trainer(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "roc_auc": roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else np.nan
    }


def get_class_weights(labels):
    counts = Counter(labels.tolist() if isinstance(labels, pd.Series) else labels)
    total = sum(counts.values())
    n_classes = len(counts)
    weights = {}
    for cls, cnt in counts.items():
        weights[cls] = total / (n_classes * cnt)

    ordered = [weights.get(i, 1.0) for i in sorted(weights.keys())]
    return torch.tensor(ordered, dtype=torch.float)


def run_transformer_model(
    model_checkpoint,
    train_df,
    val_df,
    test_df,
    run_name,
    max_length=128,
    epochs=50,
    batch_size=8,
    learning_rate=2e-5
):
    print("\n" + "=" * 70)
    print(f"TRANSFORMER TRAINING: {run_name}")
    print("=" * 70)

    train_balanced = oversample_minority_df(train_df)

    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=2
    )

    train_dataset = TextDataset(train_balanced["clean_text"].tolist(), train_balanced["label"], tokenizer, max_length=max_length)
    val_dataset = TextDataset(val_df["clean_text"].tolist(), val_df["label"], tokenizer, max_length=max_length)
    test_dataset = TextDataset(test_df["clean_text"].tolist(), test_df["label"], tokenizer, max_length=max_length)

    class_weights = get_class_weights(train_balanced["label"])

    run_output_dir = os.path.join(OUTPUT_DIR, run_name)
    os.makedirs(run_output_dir, exist_ok=True)

    training_args = TrainingArguments(
        output_dir=run_output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        learning_rate=learning_rate,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        seed=RANDOM_STATE
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics_for_trainer,
        class_weights=class_weights,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    # validation predictions for threshold tuning
    val_output = trainer.predict(val_dataset)
    val_logits = val_output.predictions
    val_prob = torch.softmax(torch.tensor(val_logits), dim=1)[:, 1].numpy()
    best_thr, _ = tune_threshold_from_probs(val_df["label"], val_prob, metric="macro_f1")
    val_pred = (val_prob >= best_thr).astype(int)

    # test predictions
    test_output = trainer.predict(test_dataset)
    test_logits = test_output.predictions
    test_prob = torch.softmax(torch.tensor(test_logits), dim=1)[:, 1].numpy()
    test_pred = (test_prob >= best_thr).astype(int)

    print(f"Best validation threshold for {run_name}: {best_thr:.2f}")
    print_evaluation(test_df["label"], test_pred, model_name=f"{run_name}_test")

    val_result = evaluate_predictions(val_df["label"], val_pred, val_prob, model_name=run_name)
    test_result = evaluate_predictions(test_df["label"], test_pred, test_prob, model_name=run_name)

    val_result.update({"split": "val", "scenario": run_name, "best_threshold": best_thr})
    test_result.update({"split": "test", "scenario": run_name, "best_threshold": best_thr})

    return pd.DataFrame([val_result, test_result]), trainer, tokenizer, best_thr

# ============================================================
# 10. AGREEMENT-BASED PSEUDO-LABELING
# ============================================================

def predict_transformer_probs(trainer, tokenizer, df, max_length=128):
    dataset = TextDataset(df["clean_text"].tolist(), df["label"].tolist(), tokenizer, max_length=max_length)
    output = trainer.predict(dataset)
    logits = output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    return probs


def agreement_pseudo_label(
    unlabeled_df,
    indobert_trainer,
    indobert_tokenizer,
    xlmr_trainer,
    xlmr_tokenizer,
    indobert_thr=0.85,
    xlmr_thr=0.85
):
    if unlabeled_df.empty:
        return pd.DataFrame(columns=list(unlabeled_df.columns) + ["pseudo_label", "conf_indobert", "conf_xlmr"]), unlabeled_df

    dummy = unlabeled_df.copy()
    if "label" not in dummy.columns:
        dummy["label"] = 0

    prob1 = predict_transformer_probs(indobert_trainer, indobert_tokenizer, dummy)
    prob2 = predict_transformer_probs(xlmr_trainer, xlmr_tokenizer, dummy)

    pred1 = (prob1 >= 0.5).astype(int)
    pred2 = (prob2 >= 0.5).astype(int)

    agree_mask = pred1 == pred2
    conf_mask = (
        ((prob1 >= indobert_thr) | (prob1 <= 1 - indobert_thr)) &
        ((prob2 >= xlmr_thr) | (prob2 <= 1 - xlmr_thr))
    )
    selected = agree_mask & conf_mask

    pseudo_df = unlabeled_df.loc[selected].copy()
    pseudo_df["pseudo_label"] = pred1[selected]
    pseudo_df["conf_indobert"] = prob1[selected]
    pseudo_df["conf_xlmr"] = prob2[selected]

    remaining_df = unlabeled_df.loc[~selected].copy().reset_index(drop=True)
    return pseudo_df.reset_index(drop=True), remaining_df


def run_ssl_transformer_round(
    labeled_df,
    unlabeled_df,
    val_df,
    test_df,
    scenario_name,
    max_rounds=1
):
    current_labeled = labeled_df.copy()
    current_unlabeled = unlabeled_df.copy()

    final_result_frames = []

    for round_idx in range(1, max_rounds + 1):
        round_name = f"{scenario_name}_ssl_round{round_idx}"
        print("\n" + "#" * 70)
        print(f"SSL ROUND: {round_name}")
        print("#" * 70)

        # train IndoBERT
        indobert_results, indobert_trainer, indobert_tokenizer, indobert_thr = run_transformer_model(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=current_labeled,
            val_df=val_df,
            test_df=test_df,
            run_name=f"indobert_{round_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5
        )

        # train XLM-R
        xlmr_results, xlmr_trainer, xlmr_tokenizer, xlmr_thr = run_transformer_model(
            model_checkpoint="FacebookAI/xlm-roberta-base",
            train_df=current_labeled,
            val_df=val_df,
            test_df=test_df,
            run_name=f"xlmr_{round_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5
        )

        final_result_frames.extend([indobert_results, xlmr_results])

        pseudo_df, current_unlabeled = agreement_pseudo_label(
            current_unlabeled,
            indobert_trainer,
            indobert_tokenizer,
            xlmr_trainer,
            xlmr_tokenizer,
            indobert_thr=max(0.80, indobert_thr),
            xlmr_thr=max(0.80, xlmr_thr)
        )

        print("Pseudo-labeled by agreement:", len(pseudo_df))
        print("Remaining unlabeled:", len(current_unlabeled))

        if pseudo_df.empty:
            break

        # Clean schema before concatenation
        pseudo_add = pseudo_df.copy()

        # Remove existing label column from unlabeled source if present
        if "label" in pseudo_add.columns:
            pseudo_add = pseudo_add.drop(columns=["label"])

        # Use the pseudo labels as the new labels
        pseudo_add["label"] = pseudo_df["pseudo_label"].astype(int)

        # Keep only columns used in current_labeled and preserve order
        pseudo_add = pseudo_add.reindex(columns=current_labeled.columns)

        current_labeled = pd.concat(
            [current_labeled, pseudo_add],
            ignore_index=True
        )

    return pd.concat(final_result_frames, ignore_index=True) if final_result_frames else pd.DataFrame()

# ============================================================
# 11. ERROR ANALYSIS
# ============================================================

def export_error_analysis(df, y_true, y_pred, y_prob, filename, top_n=50):
    out = df.copy().reset_index(drop=True)
    out["true_label"] = y_true.reset_index(drop=True).values if isinstance(y_true, pd.Series) else np.asarray(y_true)
    out["pred_label"] = np.asarray(y_pred)
    out["pred_prob_hs"] = np.asarray(y_prob)
    out["error_type"] = np.where(out["true_label"] != out["pred_label"], "error", "correct")
    out["confidence"] = np.where(out["pred_label"] == 1, out["pred_prob_hs"], 1 - out["pred_prob_hs"])

    fp = out[(out["true_label"] == 0) & (out["pred_label"] == 1)].sort_values("confidence", ascending=False).head(top_n)
    fn = out[(out["true_label"] == 1) & (out["pred_label"] == 0)].sort_values("confidence", ascending=False).head(top_n)

    with pd.ExcelWriter(os.path.join(OUTPUT_DIR, filename)) as writer:
        out.to_excel(writer, index=False, sheet_name="all_predictions")
        fp.to_excel(writer, index=False, sheet_name="false_positives")
        fn.to_excel(writer, index=False, sheet_name="false_negatives")

# ============================================================
# 12. VISUALIZATION
# ============================================================

def plot_label_distribution(data):
    counts = data["label"].value_counts().sort_index()
    labels = ["Non_HS", "HS"]

    plt.figure(figsize=(7, 5))
    plt.bar(labels, counts.values)
    plt.xlabel("Class")
    plt.ylabel("Count")
    plt.title("Label Distribution")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure_1_label_distribution.png"), dpi=300)
    plt.close()


def plot_source_distribution(data):
    counts = data["source"].value_counts()

    plt.figure(figsize=(8, 5))
    plt.bar(counts.index.astype(str), counts.values)
    plt.xticks(rotation=20, ha="right")
    plt.xlabel("Dataset Source")
    plt.ylabel("Count")
    plt.title("Merged Dataset Composition")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure_2_source_distribution.png"), dpi=300)
    plt.close()


def plot_model_comparison(results_df, split="test", metric="macro_f1"):
    df = results_df[results_df["split"] == split].copy()
    df = df.sort_values(metric, ascending=False)

    plt.figure(figsize=(14, 6))
    plt.bar(df["model"].astype(str), df[metric].astype(float))
    plt.xticks(rotation=45, ha="right")
    plt.ylabel(metric)
    plt.title(f"Model Comparison ({split.upper()} - {metric})")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"figure_3_model_comparison_{split}_{metric}.png"), dpi=300)
    plt.close()


def plot_threshold_sensitivity(results_df, split="test", metric="macro_f1"):
    df = results_df[
        (results_df["split"] == split) &
        (results_df["model"].astype(str).str.contains("fusion_|indobert|xlmr", na=False))
    ].copy()

    if "best_threshold" not in df.columns or df.empty:
        return

    # summarize one point per model
    df = df[["model", "best_threshold", metric]].drop_duplicates()

    plt.figure(figsize=(8, 5))
    plt.scatter(df["best_threshold"], df[metric])
    for _, row in df.iterrows():
        plt.annotate(str(row["model"]), (row["best_threshold"], row[metric]), fontsize=8)
    plt.xlabel("Best Validation Threshold")
    plt.ylabel(metric)
    plt.title("Threshold Sensitivity Summary")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"figure_4_threshold_sensitivity_{split}_{metric}.png"), dpi=300)
    plt.close()


def plot_learning_curve_by_fraction(results_df, metric="macro_f1"):
    df = results_df[results_df["split"] == "test"].copy()
    if "label_fraction" not in df.columns:
        return

    best_per_model_scenario = (
        df.sort_values(metric, ascending=False)
          .groupby(["model", "label_fraction"], as_index=False)
          .first()
    )

    plt.figure(figsize=(10, 6))
    for model_name, group in best_per_model_scenario.groupby("model"):
        group = group.sort_values("label_fraction")
        plt.plot(group["label_fraction"], group[metric], marker="o", label=model_name)

    plt.xlabel("Labeled Fraction")
    plt.ylabel(metric)
    plt.title("Performance vs. Labeled Fraction")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"figure_5_learning_curve_{metric}.png"), dpi=300)
    plt.close()


def plot_conf_matrix(y_true, y_pred, filename, title="Confusion Matrix"):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 5))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(2)
    plt.xticks(tick_marks, ["Non_HS", "HS"])
    plt.yticks(tick_marks, ["Non_HS", "HS"])
    plt.ylabel("True label")
    plt.xlabel("Predicted label")

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close()


def plot_roc_curve(y_true, y_prob, filename, title="ROC Curve"):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_score = roc_auc_score(y_true, y_prob)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, label=f"AUC = {auc_score:.4f}")
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=300)
    plt.close()

# ============================================================
# 13. MAIN PIPELINE
# ============================================================

def run_full_paper_pipeline():
    data = prepare_full_dataset()

    # figures from raw merged data
    plot_label_distribution(data)
    plot_source_distribution(data)

    train_df, val_df, test_df = make_train_val_test_split(data, test_size=0.2, val_size=0.1)

    low_label_scenarios = [0.05, 0.10, 0.20]
    all_results = []

    best_test_artifacts = {}

    for frac in low_label_scenarios:
        scenario_name = f"{int(frac * 100)}pct"
        print("\n" + "#" * 80)
        print(f"LOW-LABEL SCENARIO: {scenario_name}")
        print("#" * 80)

        labeled_df, unlabeled_df = make_low_label_split(train_df, labeled_fraction=frac)

        # Classical baselines
        classical_results, fitted_classical = run_classical_baselines(
            labeled_df=labeled_df,
            val_df=val_df,
            test_df=test_df,
            scenario_name=scenario_name
        )
        classical_results["label_fraction"] = frac
        all_results.append(classical_results)

        # Save best classical artifacts from ensemble
        best_classical_model = fitted_classical["fusion_logreg"]
        classical_prob = best_classical_model.predict_proba_like(test_df["clean_text"])
        classical_thr = best_classical_model.threshold_
        classical_pred = (classical_prob >= classical_thr).astype(int)

        # Transformer baselines
        indobert_results, indobert_trainer, indobert_tokenizer, indobert_thr = run_transformer_model(
            model_checkpoint="indobenchmark/indobert-base-p1",
            train_df=labeled_df,
            val_df=val_df,
            test_df=test_df,
            run_name=f"indobert_{scenario_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5
        )
        indobert_results["label_fraction"] = frac
        all_results.append(indobert_results)

        xlmr_results, xlmr_trainer, xlmr_tokenizer, xlmr_thr = run_transformer_model(
            model_checkpoint="FacebookAI/xlm-roberta-base",
            train_df=labeled_df,
            val_df=val_df,
            test_df=test_df,
            run_name=f"xlmr_{scenario_name}",
            epochs=50,
            batch_size=8,
            learning_rate=2e-5
        )
        xlmr_results["label_fraction"] = frac
        all_results.append(xlmr_results)

        # Agreement-based SSL
        ssl_transformer_results = run_ssl_transformer_round(
            labeled_df=labeled_df,
            unlabeled_df=unlabeled_df,
            val_df=val_df,
            test_df=test_df,
            scenario_name=scenario_name,
            max_rounds=1
        )
        if not ssl_transformer_results.empty:
            ssl_transformer_results["label_fraction"] = frac
            all_results.append(ssl_transformer_results)

        # Store artifacts for plots/error analysis from the last scenario or best practical one
        if frac == 0.20:
            # classical
            best_test_artifacts["classical"] = {
                "y_true": test_df["label"].copy(),
                "y_pred": classical_pred,
                "y_prob": classical_prob,
                "df": test_df.copy(),
                "name": f"classical_fusion_logreg_{scenario_name}"
            }

            # indobert
            test_dataset = TextDataset(test_df["clean_text"].tolist(), test_df["label"], indobert_tokenizer, max_length=128)
            indobert_out = indobert_trainer.predict(test_dataset)
            indobert_prob = torch.softmax(torch.tensor(indobert_out.predictions), dim=1)[:, 1].numpy()
            indobert_pred = (indobert_prob >= indobert_thr).astype(int)
            best_test_artifacts["indobert"] = {
                "y_true": test_df["label"].copy(),
                "y_pred": indobert_pred,
                "y_prob": indobert_prob,
                "df": test_df.copy(),
                "name": f"indobert_{scenario_name}"
            }

            # xlmr
            test_dataset = TextDataset(test_df["clean_text"].tolist(), test_df["label"], xlmr_tokenizer, max_length=128)
            xlmr_out = xlmr_trainer.predict(test_dataset)
            xlmr_prob = torch.softmax(torch.tensor(xlmr_out.predictions), dim=1)[:, 1].numpy()
            xlmr_pred = (xlmr_prob >= xlmr_thr).astype(int)
            best_test_artifacts["xlmr"] = {
                "y_true": test_df["label"].copy(),
                "y_pred": xlmr_pred,
                "y_prob": xlmr_prob,
                "df": test_df.copy(),
                "name": f"xlmr_{scenario_name}"
            }

    final_results = pd.concat(all_results, axis=0, ignore_index=True)
    final_results.to_csv(os.path.join(OUTPUT_DIR, "final_results.csv"), index=False)

    # Main figures
    plot_model_comparison(final_results, split="test", metric="macro_f1")
    plot_model_comparison(final_results, split="test", metric="accuracy")
    plot_threshold_sensitivity(final_results, split="test", metric="macro_f1")
    plot_learning_curve_by_fraction(final_results, metric="macro_f1")

    # Confusion matrices + ROC + error analysis
    for key, art in best_test_artifacts.items():
        plot_conf_matrix(
            art["y_true"],
            art["y_pred"],
            filename=f"figure_confusion_matrix_{key}.png",
            title=f"Confusion Matrix - {art['name']}"
        )
        plot_roc_curve(
            art["y_true"],
            art["y_prob"],
            filename=f"figure_roc_{key}.png",
            title=f"ROC Curve - {art['name']}"
        )
        export_error_analysis(
            art["df"],
            art["y_true"],
            art["y_pred"],
            art["y_prob"],
            filename=f"error_analysis_{key}.xlsx",
            top_n=50
        )

    print("\nSaved results to:", os.path.join(OUTPUT_DIR, "final_results.csv"))
    print("Saved figures and error analyses in:", OUTPUT_DIR)

    return final_results


# ============================================================
# 14. ENTRY POINT
# ============================================================

if __name__ == "__main__":
    results = run_full_paper_pipeline()

    print("\nTop TEST results by macro-F1:")
    print(
        results[results["split"] == "test"]
        .sort_values("macro_f1", ascending=False)
        [["label_fraction", "model", "accuracy", "macro_f1", "macro_precision", "macro_recall", "roc_auc", "best_threshold"]]
        .head(30)
        .to_string(index=False)
    )

Using device: cuda

DATASET: IDHSD
Shape: (713, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({0: 453, 1: 260})

DATASET: HS_572
Shape: (572, 3)
Columns: ['text', 'label', 'source']
Label distribution: Counter({1: 286, 0: 286})

DATASET: RE_13169
Shape: (13169, 4)
Columns: ['text', 'label', 'source', 'Abusive']
Label distribution: Counter({0: 7608, 1: 5561})

DATASET: Merged final
Shape: (14184, 2)
Columns: ['text', 'label']
Label distribution: Counter({0: 8251, 1: 5933})

################################################################################
LOW-LABEL SCENARIO: 5pct
################################################################################

Training classical model: fusion_svm
Best validation threshold for fusion_svm: 0.50

RESULTS: fusion_svm_5pct_test
              precision    recall  f1-score   support

           0     0.7595    0.7903    0.7746      1650
           1     0.6911    0.6521    0.6710      1187

    accuracy                     

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.621300,0.607812,0.647577,0.640522,0.728590,0.688708,0.854332
2,0.399700,0.467959,0.775330,0.774521,0.779369,0.786459,0.873882
3,0.195100,0.452836,0.814097,0.806422,0.812202,0.802974,0.885569
4,0.057100,0.580804,0.812335,0.804298,0.810836,0.800574,0.888051
5,0.015800,0.783804,0.805286,0.795852,0.805287,0.791268,0.877062


Best validation threshold for indobert_5pct: 0.44

RESULTS: indobert_5pct_test
              precision    recall  f1-score   support

           0     0.8203    0.8521    0.8359      1650
           1     0.7827    0.7405    0.7610      1187

    accuracy                         0.8054      2837
   macro avg     0.8015    0.7963    0.7985      2837
weighted avg     0.8046    0.8054    0.8046      2837

Confusion Matrix:
[[1406  244]
 [ 308  879]]

TRANSFORMER TRAINING: xlmr_5pct


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Macro Precision,Macro Recall,Roc Auc
1,0.671600,0.567748,0.769163,0.765333,0.763882,0.768764,0.808864
2,0.515400,0.501891,0.770044,0.769666,0.778592,0.784274,0.873541
3,0.394400,0.417046,0.844934,0.840317,0.841202,0.839522,0.891729
4,0.328400,0.490499,0.831718,0.830605,0.831790,0.840845,0.912246
5,0.255000,0.407053,0.853744,0.851452,0.849331,0.856244,0.912667
6,0.206500,0.487731,0.851101,0.845613,0.849681,0.842759,0.901113
7,0.165100,0.484547,0.843172,0.839353,0.838477,0.840367,0.901630


Best validation threshold for xlmr_20pct_ssl_round1: 0.70

RESULTS: xlmr_20pct_ssl_round1_test
              precision    recall  f1-score   support

           0     0.8511    0.8521    0.8516      1650
           1     0.7941    0.7928    0.7934      1187

    accuracy                         0.8273      2837
   macro avg     0.8226    0.8224    0.8225      2837
weighted avg     0.8272    0.8273    0.8273      2837

Confusion Matrix:
[[1406  244]
 [ 246  941]]


Pseudo-labeled by agreement: 6101
Remaining unlabeled: 2069



Saved results to: ./paper_outputs/final_results.csv
Saved figures and error analyses in: ./paper_outputs

Top TEST results by macro-F1:
 label_fraction                     model  accuracy  macro_f1  macro_precision  macro_recall  roc_auc  best_threshold
           0.20            indobert_20pct  0.840677  0.835072         0.838190      0.832771 0.906349            0.74
           0.20 indobert_20pct_ssl_round1  0.833627  0.829734         0.828636      0.831084 0.909234            0.44
           0.20     xlmr_20pct_ssl_round1  0.827282  0.822514         0.822591      0.822438 0.899206            0.70
           0.10            indobert_10pct  0.820938  0.816504         0.815774      0.817338 0.893309            0.66
           0.10 indobert_10pct_ssl_round1  0.817765  0.812712         0.812825      0.812601 0.895560            0.48
           0.20                xlmr_20pct  0.815650  0.812025         0.810376      0.814684 0.889070            0.64
           0.20             fusion_lo